Programming for medical imaging often involves processing large amounts of image data. Examples include image reconstruction, registration, filtering, detection, and segmentation. An image processing *pipeline* typically consists of a series of *preprocessing*, *processing*, and *postprocessing* steps. Many of these steps are computationally intensive and can therefore take considerable time to complete.

If this slows down your work too much, it may be worth investing time in identifying and addressing *bottlenecks*. However, it is important to take a holistic view of your workflow. Research involves both manual tasks (such as collecting data, reading and writing papers, and interacting with patients) and automated tasks, such as image processing. Waiting hours or even days for an image processing pipeline to finish may not be a limiting factor if you can use that time productively for other tasks. In contrast, having to wait ten minutes for a clinical image processing pipeline to finish could be a critical bottleneck when a patient is in an acute, life-threatening condition.

## Measuring Runtime

When the image processing pipline is a time limiting factor, then we should first pinpoint the bottleneck(s) in the pipeline. We should first measure the *runtime* for each major step in our processing pipeline. If there are three steps that take 2 minutes, 25 minutes, and 30 seconds, respectively, then we should focus on improving the performance of the second step. In Python, runtime can be measured with the `time` module. This module provides the `perf_counter()` function, which can be used to reliably measure the runtime of a process. Calling `perf_counter()` returns a floating point value in seconds which on itself has no meaning, but the difference between calls is a reliable measure of the time elapsed:

In [15]:
import time


start = time.perf_counter()
time.sleep(1.23)  # The 'sleep' function waits for a given amount of seconds.
elapsed = time.perf_counter() - start

print(f"Processing took {elapsed:.3f} seconds")

Processing took 1.231 seconds


N.b. `time.time()` also returns a timestamp in seconds, but this function is affected by the system clock, whereas `time.perf_counter()` is not.

## Timescales of Computer Processes

Once we have identified the major time-consuming step(s) in our processing pipeline, we can investigate them in more detail. Understanding the typical timescales of different operations in a computer helps us make educated guesses about where to look for potential bottlenecks.

### CPU Instructions

Let's start with some of the fastest operations: basic operations that a CPU can do. These include comparisons, addition, subtraction, multiplication, division, and taking square roots. The fundamental unit of time inside a CPU is a *clock cycle*. Modern CPUs typically operate at frequencies of several gigahertz (GHz). At 1 GHz, one clock cycle takes one nanosecond (a billionth of a second). On modern CPUs, simple arithmetic instructions such as addition, subtraction, and multiplication can often be performed with a throughput of one or more instructions per clock cycle. Division and calculating square roots are generally more expensive. And exponentials, floating point powers and logarithms are far more expensive. Nevertheless, these instructions typically take place on a nanosecond timescale. This means that if the CPU was the only limiting factor, we could multiply all values in a 1000×1000×1000 voxel image in just one second.

### Cache Memory and RAM

Perhaps surprisingly, the speed at which data can be delivered to and from the CPU is limited by the physical distance between the CPU and the memory in which this data is stored. The closer to the CPU, the faster memory can be, but the physical size is limited. For this reason, computer memory is layered.

At the highest level, closest to the CPU core, there is so-called level 1 cache memory, or *L1*, with a size of a few tens of kilobytes (just enough for several thousand floating point numbers) per core. A CPU core can read and write to this cache at almost the same speed as performing basic instructions. Level 2 cache is usually in the order of one megabyte (a 512×512 image), but roughly 4 times slower, and level 3 cache can be a few tens of megabytes large (a small 3D image) and is roughly 10 times as slow. These memory caches are all on the same chip as the CPU cores. The L3 cache is shared between all cores on a multicore CPU, whereas each core has its own L1 and L2 cache memory. Note that these caches do not only need to store our input data, but also the output, the algorithm itself, and all kinds of information to just run the computer.

![L1, L2, and L3 cache on a 4-core CPU (from: https://superuser.com/questions/196143/)](images/cpu-cache.png)

Further away from the CPU, on separate chips, but still directly connected to the memory controller on the CPU, is the computer's random access memory (RAM). This memory is much larger, in the order of tens of gigabytes in your computer (enough to contain a dozen large 3D images, or a couple of 3D times series) and up to a terabyte in a compute-server. RAM memory is, however, approximately 50 times slower than L1 cache memory.

When your algorithm requires a pixel value in of an image, the CPU first checks whether the required data is available in its caches. If not, it retrieves the data from a lower level of the memory hierarchy. It is called a *cache miss* when the requested data is absent from a particular cache. A miss in L1 does not necessarily mean RAM must be accessed; the data may be in L2 or L3. Furthermore, CPUs do not fetch single values from RAM whenever they need a value. Data is transferred in blocks called *cache lines*, which can contain e.g. 8 adjacent 64-bit floating point numbers or 16 adjacent 32-bit floating point numbers that are transferred at once. Modern CPUs can also detect predictable access patterns, such as reading consecutive memory addresses, and prefetch upcoming cache lines before the program requests them.

These mechanisms explain why memory access patterns can have a major impact on performance. When accessing pixel values as a contiguous array, we should always try to access them in memory order. For a conventional row-major image, this means processing consecutive values along each row before moving to the next row. Doing so makes better use of each cache line, helps hardware prefetching, and can reduce the time spent waiting for data.

Now that we understand how memory access can limit an algorithm’s performance, it becomes clear why choosing the smallest data type that provides sufficient precision can be important. For example, an image stored using 32-bit floating-point values requires half as much memory as the same image stored using 64-bit floating-point values. This reduces memory usage and the amount of data that needs to be transferred between memory and the CPU. As a result, processing the image may be significantly faster.

### Storage Devices

Unlike volatile RAM, which loses its contents when the computer is switched off, an SSD hard drive, network storage, or cloud storage will retain data. This is known as *non-volatile storage*. However, accessing data from these storage devices is much slower than accessing data in RAM. SSD access latency can be tens to hundreds of times greater than RAM access latency. Network and cloud storage can introduce even greater delays, sometimes thousands of times greater, depending on the storage service. This is why we should avoid unnecessary reading and writing to storage devices. For example, repeatedly saving intermediate results to SSD and loading them again can significantly increase the total processing time.

There is another potential performance problem: what happens when your program uses more RAM than is available? Operating systems can move memory contents to disk, which allows programs to continue running even when physical RAM is insufficient. If an image processing pipeline repeatedly needs data that has been moved to disk, the operating system may have to move that data back into RAM while moving other data out. You can imagine that this severely degrades performance. Depending on the algorithm, processing data in smaller chunks may help reduce such issues.

> **External course material**
>
> This YouTube video provides an intuitive understanding of computer timescales by mapping them onto human timescales:
>
> * <https://www.youtube.com/watch?v=PpaQrzoDW2I>

## Monitoring CPU and Memory Use

All operating systems have an app for monitoring CPU and memory use. These apps show a list of processes, along with their CPU and memory use. Processes are running instances of programs that you, or the operating system itself started. For Python programs, the process is just called `python`.

These are the monitoring apps that are available on each operating system:

* **Windows**: [Task Manager](https://en.wikipedia.org/wiki/Task_Manager_(Windows))
* **macOS**: [Activity Monitor](https://support.apple.com/guide/activity-monitor/welcome/mac)
* **Linux**: System Monitor (Gnome, KDE, and other desktops have their own version)

When you are new to this app, then open it and browse around to see what information it shows.

A monitoring tool like this can help us to get clues on what's going when an image processing pipeline is running. For example:

* The CPU is heavily utilized: The algorithm is likely to be CPU-bound; computation is the bottleneck.
* The CPU is moderately utilized, and there is memory available: The algorithm could be memory-bound; throughput is the bottleneck.
* The RAM is nearly full and disk activity high: The processing pipeline might be using too much memory.